# DST Domestic Workflow v1.2 — Macro Functions

Auto-generated Fabric equivalents for 5 Alteryx macros.
Each macro is translated to a callable PySpark function.

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

In [ ]:
def macro_main_macro_datahubpush_v2(
    data_records,
    collection_name,
    template_name,
    recordset_name,
    client_id,
    client_secret,
    base_url,
    scope="urn:tr:onesource:auth:api:datahub"
):
    """
    Fabric equivalent of Alteryx macro: Main_Macro_DataHubPush_V2
    Classification: notebook_complex
    Original tools: 124
    Type: API/HTTP macro — Full 13-step DataHub push orchestration.

    Steps:
        1. OAuth2 token
        2. Create ingest file
        3. Upload file data (CSV)
        4. Check upload status
        5. Get template by name
        6. Trigger processing job
        7. Poll job status (calls IterativeMacro_6)
        8. Get collection by name
        9. Unpublish collection
        10. Get & delete old recordset
        11. Create new recordset with data
        12. Publish collection
        13. Confirm publish (calls IterativeMacro_12)

    Parameters:
        data_records: list[dict] — EDP output data to push
        collection_name: Target collection name
        template_name: Template/schematic name for field mapping
        recordset_name: Name for the new recordset (e.g., "EDP_Domestic_2025_Q1")
        client_id: OAuth2 client ID
        client_secret: OAuth2 client secret
        base_url: DataHub API base URL
        scope: OAuth2 scope

    Returns:
        dict: {"success": bool, "steps": [...], "collection_id": str, "recordset_id": str}
    """
    import requests
    import csv
    import io

    steps_log = []

    def log_step(step_num, name, status, detail=""):
        steps_log.append({"step": step_num, "name": name, "status": status, "detail": detail})

    # =========================================================================
    # STEP 1: OAuth2 Token
    # =========================================================================
    resp = requests.post(
        f"{base_url}/auth/token",
        data=f"client_id={client_id}&client_secret={client_secret}&grant_type=client_credentials&scope={scope}"
    )
    resp.raise_for_status()
    access_token = resp.json()["access_token"]
    log_step(1, "OAuth2 Token", "success", f"token={access_token[:12]}...")

    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}

    # =========================================================================
    # STEP 2: Create ingest file
    # =========================================================================
    resp = requests.post(
        f"{base_url}/api/v1/ingestfiles",
        json={"filename": f"{recordset_name}.csv"},
        headers=headers
    )
    resp.raise_for_status()
    ingest_file_id = resp.json()["id"]
    log_step(2, "Create Ingest File", "success", f"id={ingest_file_id}")

    # =========================================================================
    # STEP 3: Upload file data (convert records to CSV)
    # =========================================================================
    if data_records:
        output = io.StringIO()
        writer = csv.DictWriter(output, fieldnames=data_records[0].keys())
        writer.writeheader()
        writer.writerows(data_records)
        csv_data = output.getvalue()
    else:
        csv_data = ""

    resp = requests.post(
        f"{base_url}/api/v1/ingestfiles/{ingest_file_id}/upload",
        data=csv_data.encode("utf-8"),
        headers={"Authorization": f"Bearer {access_token}", "Content-Type": "text/csv"}
    )
    resp.raise_for_status()
    log_step(3, "Upload File Data", "success", f"bytes={len(csv_data)}")

    # =========================================================================
    # STEP 4: Check upload status
    # =========================================================================
    resp = requests.get(f"{base_url}/api/v1/ingestfiles/{ingest_file_id}", headers=headers)
    resp.raise_for_status()
    upload_status = resp.json()["status"]
    log_step(4, "Check Upload Status", "success", f"status={upload_status}")

    # =========================================================================
    # STEP 5: Get template by name
    # =========================================================================
    resp = requests.get(
        f"{base_url}/api/v1/templates",
        params={"filter": f"name eq '{template_name}'"},
        headers=headers
    )
    resp.raise_for_status()
    templates = resp.json().get("value", [])
    template_id = templates[0]["id"] if templates else None
    log_step(5, "Get Template", "success" if template_id else "warning", f"template_id={template_id}")

    # =========================================================================
    # STEP 6: Trigger processing job
    # =========================================================================
    resp = requests.post(
        f"{base_url}/api/v1/ingestfiles/{ingest_file_id}/process",
        json={"template_id": template_id},
        headers=headers
    )
    resp.raise_for_status()
    job_id = resp.json()["id"]
    log_step(6, "Trigger Processing", "success", f"job_id={job_id}")

    # =========================================================================
    # STEP 7: Poll job status (IterativeMacro_6)
    # =========================================================================
    job_result = macro_iterativemacro_6_confirmjobstatus(
        job_id=job_id,
        access_token=access_token,
        base_url=base_url,
        max_attempts=30,
        poll_interval=2
    )
    log_step(7, "Confirm Job Status", job_result["status"], f"attempts={job_result['attempts']}")
    if job_result["status"] != "complete":
        return {"success": False, "steps": steps_log, "error": "Job did not complete"}

    # =========================================================================
    # STEP 8: Get collection by name
    # =========================================================================
    resp = requests.get(
        f"{base_url}/api/v1/collections",
        params={"filter": f"name eq '{collection_name}'"},
        headers=headers
    )
    resp.raise_for_status()
    collections = resp.json().get("value", [])
    if not collections:
        log_step(8, "Get Collection", "error", f"Collection '{collection_name}' not found")
        return {"success": False, "steps": steps_log, "error": f"Collection '{collection_name}' not found"}
    collection_id = collections[0]["id"]
    log_step(8, "Get Collection", "success", f"id={collection_id}")

    # =========================================================================
    # STEP 9: Unpublish collection
    # =========================================================================
    resp = requests.post(f"{base_url}/api/v1/collections/{collection_id}/unpublish", headers=headers)
    resp.raise_for_status()
    log_step(9, "Unpublish Collection", "success")

    # =========================================================================
    # STEP 10: Get existing recordsets and delete them
    # =========================================================================
    resp = requests.get(f"{base_url}/api/v1/collections/{collection_id}/recordsets", headers=headers)
    resp.raise_for_status()
    existing_recordsets = resp.json().get("value", [])
    for rs in existing_recordsets:
        requests.delete(
            f"{base_url}/api/v1/collections/{collection_id}/recordsets/{rs['id']}",
            headers=headers
        )
    log_step(10, "Delete Old Recordsets", "success", f"deleted={len(existing_recordsets)}")

    # =========================================================================
    # STEP 11: Create new recordset with data
    # =========================================================================
    resp = requests.post(
        f"{base_url}/api/v1/collections/{collection_id}/recordsets",
        json={
            "name": recordset_name,
            "template_name": template_name,
            "record_count": len(data_records),
            "records": data_records
        },
        headers=headers
    )
    resp.raise_for_status()
    new_recordset_id = resp.json()["id"]
    log_step(11, "Create Recordset", "success", f"id={new_recordset_id}, records={len(data_records)}")

    # =========================================================================
    # STEP 12: Publish collection
    # =========================================================================
    resp = requests.post(f"{base_url}/api/v1/collections/{collection_id}/publish", headers=headers)
    resp.raise_for_status()
    log_step(12, "Publish Collection", "success")

    # =========================================================================
    # STEP 13: Confirm publish status (IterativeMacro_12)
    # =========================================================================
    publish_result = macro_iterativemacro_12_confirmpublisheddataset(
        collection_id=collection_id,
        access_token=access_token,
        base_url=base_url,
        max_attempts=20,
        poll_interval=2
    )
    log_step(13, "Confirm Published", publish_result["status"], f"attempts={publish_result['attempts']}")

    return {
        "success": publish_result["status"] == "published",
        "steps": steps_log,
        "collection_id": collection_id,
        "recordset_id": new_recordset_id,
        "records_pushed": len(data_records)
    }

In [ ]:
def macro_edp_macro_domestic_mod(df):
    """
    Fabric equivalent of Alteryx macro: EDP_Macro_Domestic_Mod
    Classification: notebook_parameterized
    Original tools: 35
    Type: Batch macro (parameterized execution)
    """
    from pyspark.sql import functions as F

    # Filter step (from macro)
    # result_df = df.filter(F.col('column') == 'value')
    # Formula/calculation step (from macro)
    # result_df = df.withColumn('new_col', F.expr('expression'))
    # Join step (from macro)
    # result_df = df.join(other_df, on='key', how='inner')
    # Aggregation step (from macro)
    # result_df = df.groupBy('col').agg(F.sum('val'))
    # TODO: Implement full transformation logic
    return df

In [ ]:
def macro_data_hub_get_collection_data(collection_name, instance_id, access_token, base_url, page_size=500):
    """
    Fabric equivalent of Alteryx macro: Data Hub - Get Collection Data
    Classification: notebook_complex
    Original tools: 46
    Type: API/HTTP macro — paginated read of all records from a named collection.

    Parameters:
        collection_name: Name of the DataHub collection to read
        instance_id: X-OneSourceInstanceId header value
        access_token: OAuth2 Bearer token
        base_url: DataHub API base URL
        page_size: Records per page (default 500)

    Returns:
        list[dict]: All records from the collection (paginated retrieval)
    """
    import requests

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "X-OneSourceInstanceId": instance_id,
    }

    # Step 1: Find collection by name
    resp = requests.get(
        f"{base_url}/api/v1/collections",
        params={"filter": f"name eq '{collection_name}'"},
        headers=headers
    )
    resp.raise_for_status()
    collections = resp.json().get("value", [])
    if not collections:
        raise ValueError(f"Collection '{collection_name}' not found")

    collection_id = collections[0]["id"]

    # Step 2: Paginated read
    all_records = []
    offset = 0
    while True:
        resp = requests.get(
            f"{base_url}/api/v1/collections/{collection_id}/data",
            params={"offset": offset, "limit": page_size},
            headers=headers
        )
        resp.raise_for_status()
        page = resp.json()
        records = page.get("value", [])
        all_records.extend(records)

        total = page.get("total_count", 0)
        offset += len(records)
        if offset >= total or len(records) == 0:
            break

    return all_records

In [ ]:
def macro_iterativemacro_12_confirmpublisheddataset(collection_id, access_token, base_url, max_attempts=20, poll_interval=15):
    """
    Fabric equivalent of Alteryx macro: IterativeMacro_12_ConfirmPublishedDataset
    Classification: notebook_complex
    Original tools: 17
    Type: Iterative macro — polls collection publish status until published.

    Parameters:
        collection_id: The collection UUID to check
        access_token: OAuth2 Bearer token
        base_url: DataHub API base URL
        max_attempts: Maximum polling iterations (default 20)
        poll_interval: Seconds between polls (default 15)

    Returns:
        dict: {"status": "published"|"failed"|"timeout", "attempts": N, "response": last_response}
    """
    import requests
    import time

    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
    url = f"{base_url}/api/v1/collections/{collection_id}/status"

    for attempt in range(1, max_attempts + 1):
        resp = requests.get(url, headers=headers)
        resp.raise_for_status()
        data = resp.json()

        status = data.get("status", "").lower()
        if status == "published":
            return {"status": "published", "attempts": attempt, "response": data}
        elif status in ("failed", "error"):
            return {"status": "failed", "attempts": attempt, "response": data}

        if attempt < max_attempts:
            time.sleep(poll_interval)

    return {"status": "timeout", "attempts": max_attempts, "response": data}

In [ ]:
def macro_iterativemacro_6_confirmjobstatus(job_id, access_token, base_url, max_attempts=30, poll_interval=10):
    """
    Fabric equivalent of Alteryx macro: IterativeMacro_6_ConfirmJobStatus
    Classification: notebook_complex
    Original tools: 14
    Type: Iterative macro — polls job status until complete or failed.

    Parameters:
        job_id: The processing job ID returned by the process endpoint
        access_token: OAuth2 Bearer token
        base_url: DataHub API base URL
        max_attempts: Maximum number of polling iterations (default 30)
        poll_interval: Seconds between polls (default 10)

    Returns:
        dict: {"status": "complete"|"failed"|"timeout", "attempts": N, "response": last_response}
    """
    import requests
    import time

    headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
    url = f"{base_url}/api/v1/jobs/{job_id}"

    for attempt in range(1, max_attempts + 1):
        resp = requests.get(url, headers=headers)
        resp.raise_for_status()
        data = resp.json()

        status = data.get("status", "").lower()
        if status == "complete":
            return {"status": "complete", "attempts": attempt, "response": data}
        elif status in ("failed", "error"):
            return {"status": "failed", "attempts": attempt, "response": data}

        if attempt < max_attempts:
            time.sleep(poll_interval)

    return {"status": "timeout", "attempts": max_attempts, "response": data}

## Macro Function Registry

| Macro | Function | Classification |
|-------|----------|----------------|
| Main_Macro_DataHubPush_V2 | `macro_main_macro_datahubpush_v2()` | notebook_complex |
| EDP_Macro_Domestic_Mod | `macro_edp_macro_domestic_mod()` | notebook_parameterized |
| Data Hub - Get Collection Data | `macro_data_hub_get_collection_data()` | notebook_complex |
| IterativeMacro_12_ConfirmPublishedDataset | `macro_iterativemacro_12_confirmpublisheddataset()` | notebook_complex |
| IterativeMacro_6_ConfirmJobStatus | `macro_iterativemacro_6_confirmjobstatus()` | notebook_complex |